In [1]:
import pandas as pd
import glob
from pathlib import Path

In [5]:
imputed_dir = Path("imputed_ohe_data")
simulated_dir = Path("simulated_data")
target_dir = Path("imputed_data")

In [8]:
for imputed_file in imputed_dir.glob("*.csv"):
    # Build corresponding csv path with same stem (filename without extension)
    simulated_file = simulated_dir / str(imputed_file).split("/")[1]
    simulated_file = (str(simulated_file)[:-3])+"tsv"
    
    imputed_df = pd.read_csv(imputed_file)
    imputed_df.set_index("ID", inplace=True)
    
    simulated_df = pd.read_csv(simulated_file, sep="\t", index_col=0)
    simulated_df.drop(columns=['type'], inplace=True)
    simulated_df = simulated_df.T.copy()
    simulated_df.replace({-99999.0 : pd.NA}, inplace=True)
    
    # Subset to OHE variables.
    ohe_cols = [col for col in imputed_df.columns if col.startswith("ohe")]
    non_ohe_cols = set(imputed_df.columns) - set(ohe_cols)
    ohe_df = imputed_df[ohe_cols].copy()
    
    ohe_vars = set([col[4:-4] for col in ohe_cols])
    var_to_columns = {var : [col for col in ohe_cols if (var in col)] for var in ohe_vars}
        
    trafo_df = pd.DataFrame(index=ohe_df.index, columns=list(ohe_vars))

    # Iterate over all OHE variables and choose maximum imputed value as category.
    for var in ohe_vars:
        for sample in trafo_df.index:
            # Check if sample-var pair has been NA in input data.
            if pd.isna(simulated_df.loc[sample, var]):
                # Search max value in OHE imputed file.
                var_categories = var_to_columns[var]
                col_values = {col : imputed_df.at[sample, col] for col in var_categories}
                max_col = max(col_values, key=col_values.get)
                max_category = float(max_col[-3:])
                # Set best category into imputed dataframe.
                trafo_df.loc[sample, var] = max_category
            else:
                # Same procedure as in if-case: actual category has value 1, all other 0.
                # Search max value in OHE imputed file.
                var_categories = var_to_columns[var]
                col_values = {col : imputed_df.at[sample, col] for col in var_categories}
                max_col = max(col_values, key=col_values.get)
                max_category = float(max_col[-3:])
                # Set best category into imputed dataframe.
                trafo_df.loc[sample, var] = max_category

    # Combine imputed df with ohe-transformed imputed df.
    final_df = pd.concat([imputed_df[list(non_ohe_cols)], trafo_df], axis=1)
    final_df['ID'] = final_df.index
    target_file = target_dir / str(imputed_file).split("/")[1]
    final_df.to_csv(target_file, index=False)